# Lab | Week 7 | Day 5 | Micromanage your agents
## John Adams

## Part 1: Setup & Document Selection
### Step 2: Set Up LangSmith Environment

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Loads keys/tokens from .env

import os

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com" # uses LangSmith EU endpoint
os.environ["LANGSMITH_PROJECT"]  = "ironhack-lab-micromanage-your-agents-dataset"


In [2]:
from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI

client = Client()
openai_client = wrap_openai(OpenAI())


In [ ]:
# Verify connection
datasets = list(client.list_datasets())
print(f"Connected to LangSmith. {len(datasets)} existing dataset(s) in this workspace.")


## Part 2: Dataset Creation
### Step 3: Extract and Structure Your Examples

In [ ]:
import pandas as pd

# source: LeadForge Lead Scoring Intro v1, Kaggle, derelictpanda/leadforge-lead-scoring-intro-v1
raw = pd.read_csv("Data/lead_scoring.csv")

INPUT_COLS = [
    "industry", "region", "employee_band", "lead_source",
    "touch_count", "session_count", "pricing_page_views", "demo_page_views",
    "days_since_first_touch", "days_since_last_touch", "opportunity_estimated_acv",
]
OUTPUT_COL = "converted_within_90_days"
META_COLS = ["lead_id", "account_id"]

print(f"Full dataset: {len(raw)} leads")
print(raw["lead_source"].value_counts())


Full dataset: 5000 leads
lead_source
inbound_marketing    2274
sdr_outbound         1757
partner_referral      969
Name: count, dtype: int64


In [ ]:
# Stratified sample, 3 rows per lead_source x converted combination, 18 examples total
sample = (
    raw.groupby(["lead_source", "converted_within_90_days"])
       .apply(lambda g: g.sample(n=3, random_state=42), include_groups=False)
       .reset_index()
       .drop(columns=["level_2"])
)

print(f"Sampled {len(sample)} examples")
sample[["lead_id", "lead_source", "converted_within_90_days"]]


Sampled 18 examples


,lead_id,lead_source,converted_within_90_days
0,lead_001548,inbound_marketing,False
1,lead_003911,inbound_marketing,False
2,lead_000535,inbound_marketing,False
3,lead_000500,inbound_marketing,True
4,lead_004765,inbound_marketing,True
5,lead_003224,inbound_marketing,True
6,lead_003578,partner_referral,False
7,lead_003506,partner_referral,False
8,lead_003674,partner_referral,False
9,lead_000174,partner_referral,True


In [ ]:
# Structure each row as a LangSmith example
examples = []
for _, row in sample.iterrows():
    examples.append({
        "inputs": {col: row[col] for col in INPUT_COLS},
        "outputs": {"converted_within_90_days": bool(row[OUTPUT_COL])},
        "metadata": {col: row[col] for col in META_COLS},
    })

print(f"Structured {len(examples)} examples")
examples[0]


Structured 18 examples


{'inputs': {'industry': 'manufacturing',
  'region': 'UK',
  'employee_band': '500-999',
  'lead_source': 'inbound_marketing',
  'touch_count': 8.0,
  'session_count': 1.0,
  'pricing_page_views': 1.0,
  'demo_page_views': 1.0,
  'days_since_first_touch': 28.297033914864024,
  'days_since_last_touch': 5.292759927307445,
  'opportunity_estimated_acv': nan},
 'outputs': {'converted_within_90_days': False},
 'metadata': {'lead_id': 'lead_001548', 'account_id': 'acct_000047'}}

### Step 4: Create LangSmith Dataset

In [ ]:
DATASET_NAME = "ironhack-lab-micromanage-your-agents-lead-scoring"
DATASET_DESCRIPTION = (
    "18 B2B SaaS leads sampled from LeadForge Lead Scoring Intro v1 (Kaggle). Inputs are a lead's 30-day engagement snapshot, outputs are ground-truth converted_within_90_days. Stratified across lead_source and outcome."
)

# Create dataset, upload examples, and skip if it already exists
existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(DATASET_NAME, description=DATASET_DESCRIPTION)
    client.create_examples(
        inputs=[ex["inputs"] for ex in examples],
        outputs=[ex["outputs"] for ex in examples],
        metadata=[ex["metadata"] for ex in examples],
        dataset_id=dataset.id,
    )
    print(f"Created dataset with {len(examples)} examples")
else:
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Dataset {DATASET_NAME!r} already exists, skipping upload")


Created dataset with 18 examples


In [8]:
# Verify
uploaded = list(client.list_examples(dataset_id=dataset.id))
print(f"Dataset {DATASET_NAME!r} has {len(uploaded)} examples")
print(f"View at: {client._host_url}/datasets/{dataset.id}")


Dataset 'ironhack-lab-micromanage-your-agents-lead-scoring' has 18 examples
View at: https://eu.smith.langchain.com/datasets/b294578b-5f9a-427c-83b9-542b448a3780


## Part 3: Evaluation Setup
### Step 5: Implement Target Function

In [9]:
LEAD_SCORING_SYSTEM_PROMPT = """\
You are a B2B SaaS marketing ops analyst scoring inbound leads. Given a lead's 30-day engagement snapshot, predict whether the lead will convert to a paying customer within 90 days.

Reply with only "true" or "false", nothing else.
"""


@traceable(name="lead-conversion-prediction")
def predict_conversion(inputs: dict) -> dict:
    lead_summary = "\n".join(f"{k}: {v}" for k, v in inputs.items())

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": LEAD_SCORING_SYSTEM_PROMPT},
                {"role": "user", "content": lead_summary},
            ],
            max_tokens=5,
        )
        raw = response.choices[0].message.content.strip().lower()
        prediction = raw.startswith("true")
    except Exception as e:
        print(f"target function error: {e}")
        prediction = False

    return {"converted_within_90_days": prediction}


In [10]:
# test on 3 examples before running the full dataset
for ex in examples[:3]:
    result = predict_conversion(ex["inputs"])
    print(f"predicted: {result['converted_within_90_days']!s:5}  actual: {ex['outputs']['converted_within_90_days']}")


predicted: False  actual: False
predicted: False  actual: False
predicted: False  actual: False


### Step 6: Set Up Evaluators

In [13]:
from openevals.llm import create_llm_as_judge

# tighter binary correctness prompt, our task is exact True/False match, not open-ended QA
LEAD_CORRECTNESS_PROMPT = """You are grading a binary lead-conversion prediction.

The output field converted_within_90_days is either true or false.
Score 1 if the output value exactly matches the reference value, otherwise score 0.
Do not consider phrasing, reasoning, or anything other than the boolean value itself.

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<reference_outputs>
{reference_outputs}
</reference_outputs>"""

_judge = create_llm_as_judge(prompt=LEAD_CORRECTNESS_PROMPT, model="openai:gpt-4o-mini", feedback_key="correctness")


def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return _judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)


## Part 4: Run Evaluation
### Step 7: Execute Evaluation Experiment

In [14]:
results = client.evaluate(
    predict_conversion,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o-mini-lead-scoring",
    max_concurrency=2,
)
print(results)


View the evaluation results for experiment: 'gpt-4o-mini-lead-scoring-d70eabbf' at:
https://eu.smith.langchain.com/o/30e98d46-c47a-47c0-ad74-25a7832452c5/datasets/b294578b-5f9a-427c-83b9-542b448a3780/compare?selectedSessions=9c9defca-c69a-41b1-89ff-3fc2f620f30b




0it [00:00, ?it/s]

<ExperimentResults gpt-4o-mini-lead-scoring-d70eabbf>


## Optional Part 2: Cost & Performance Optimization
### Step 14: Run A/B Comparison

In [15]:
@traceable(name="lead-conversion-prediction-gpt4o")
def predict_conversion_gpt4o(inputs: dict) -> dict:
    lead_summary = "\n".join(f"{k}: {v}" for k, v in inputs.items())

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            temperature=0,
            messages=[
                {"role": "system", "content": LEAD_SCORING_SYSTEM_PROMPT},
                {"role": "user", "content": lead_summary},
            ],
            max_tokens=5,
        )
        raw = response.choices[0].message.content.strip().lower()
        prediction = raw.startswith("true")
    except Exception as e:
        print(f"target function error: {e}")
        prediction = False

    return {"converted_within_90_days": prediction}


In [16]:
results_4o = client.evaluate(
    predict_conversion_gpt4o,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o-lead-scoring",
    max_concurrency=2,
)
print(results_4o)


View the evaluation results for experiment: 'gpt-4o-lead-scoring-62598ef8' at:
https://eu.smith.langchain.com/o/30e98d46-c47a-47c0-ad74-25a7832452c5/datasets/b294578b-5f9a-427c-83b9-542b448a3780/compare?selectedSessions=e7c533b2-ceb5-47d1-9860-c86574d142e2




0it [00:00, ?it/s]

<ExperimentResults gpt-4o-lead-scoring-62598ef8>
